# Adaptive v2 Kaggle GPU Run

This notebook pulls the versioned `adaptive_v2` code, verifies the Kaggle GPU, runs one seed for 50 rounds, and saves a local output log, JSON results, and per-family charts.

In [24]:
import json
import os
import subprocess
import sys
from pathlib import Path

project_dir = Path("/kaggle/working/mastercard_hackathon")
repository_url = "https://github.com/keshav-0210/mastercard_hackathon.git"
if not project_dir.exists():
    subprocess.run(["git", "clone", repository_url, str(project_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]

model_dir = Path("/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local")
gguf_files = sorted(model_dir.glob("*.gguf"))
if len(gguf_files) < 2:
    raise FileNotFoundError(f"Expected two Qwen GGUF shards under {model_dir}, found: {gguf_files}")
model_path = next(path for path in gguf_files if "00001-of-00002" in path.name)

sys.path.insert(0, str(project_dir / "src"))
os.chdir(project_dir)
os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
os.environ["GPU_LAYERS"] = "-1"

import torch
print("Project:", project_dir)
print("Model shards:", len(gguf_files))
print("Model path:", model_path)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Enable GPU in Kaggle before running this notebook"
print("Kaggle GPU check: OK")

Updating 69bf684..ec644f3
Fast-forward
 src/mastercard_defence/loop.py | 10 ++++++++++
 1 file changed, 10 insertions(+)
Project: /kaggle/working/mastercard_hackathon
Model shards: 2
Model path: /kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
CUDA available: True
GPU: Tesla T4
Kaggle GPU check: OK


From https://github.com/keshav-0210/mastercard_hackathon
   69bf684..ec644f3  main       -> origin/main


In [27]:
%pip install --no-cache-dir ctgan

import ctgan
print("CTGAN version:", getattr(ctgan, "__version__", "installed"))
print("CTGAN installation: OK")

Note: you may need to restart the kernel to use updated packages.
CTGAN version: 0.12.1
CTGAN installation: OK


In [28]:
%pip install --no-cache-dir --prefer-binary --index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --extra-index-url https://pypi.org/simple llama-cpp-python

from llama_cpp import llama_supports_gpu_offload
llama_gpu_available = bool(llama_supports_gpu_offload())
print("llama.cpp GPU offload support:", llama_gpu_available)
if not llama_gpu_available:
    print("Qwen GPU offload unavailable; CTGAN GPU will still be used and the smoke/full run will use HeuristicAgents.")
else:
    print("llama.cpp CUDA installation: OK")

Looking in indexes: https://abetlen.github.io/llama-cpp-python/whl/cu124, https://pypi.org/simple
Note: you may need to restart the kernel to use updated packages.
llama.cpp GPU offload support: True
llama.cpp CUDA installation: OK


In [ ]:
import tempfile
from datetime import datetime, timezone

from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

RUN_ROUNDS = 50
FAMILY_PROBE_SIZE = 20
FAMILY_PROBE_BACKEND = "procedural"

config = load_config(str(project_dir / "config" / "default.yaml"))
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
config["paths"]["memory_db"] = str(project_dir / "artifacts" / f"adaptive_v2_memory_{run_stamp}.sqlite")
config["generator_backend"] = "ctgan"
config["generator_epochs"] = 3
config["generator_training_attack_size"] = 40
config["generator_training_reference_size"] = 400
config["detector_mode"] = "static"
config["family_probe_backend"] = FAMILY_PROBE_BACKEND
config["family_probe_size"] = FAMILY_PROBE_SIZE
config["pipeline"]["rounds"] = RUN_ROUNDS
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80

agents = QwenAgents(config, llm=SharedLocalLLM(config))
loop = ClosedLoop(config, agents=agents)
print("Adaptive v2 configuration ready")
print("Agent backend:", type(agents).__name__)
print("Generator:", config["generator_backend"])
print("Generator epochs:", config["generator_epochs"])
print("Detector:", config["detector_mode"])
print("Family probe backend:", config["family_probe_backend"])
print("Family probe size:", config["family_probe_size"])
print("Seeds: 1")
print("Rounds:", RUN_ROUNDS)
print("Run timestamp:", run_stamp)

Adaptive v2 configuration ready
Agent backend: QwenAgents
Generator: ctgan
Generator epochs: 3
Detector: static
Seeds: 1
Rounds: 50
Run timestamp: 20260825T143148Z


In [30]:
import tempfile

smoke_config = load_config(str(project_dir / "config" / "default.yaml"))
smoke_config["paths"]["memory_db"] = tempfile.mktemp(suffix="_adaptive_v2_smoke.sqlite")
smoke_config["generator_backend"] = "ctgan"
smoke_config["generator_epochs"] = 1
smoke_config["generator_training_attack_size"] = 5
smoke_config["generator_training_reference_size"] = 50
smoke_config["detector_mode"] = "static"
smoke_config["pipeline"]["synthetic_transactions"] = 80
smoke_config["pipeline"]["max_generated_attacks"] = 10

smoke_loop = ClosedLoop(smoke_config, agents=agents)
try:
    smoke_results = smoke_loop.run(rounds=1, seed=20260821)
finally:
    smoke_loop.close()

assert len(smoke_results) == 1
assert smoke_results[0]["round"] == 1
assert len(smoke_results[0]["detection"]["all_family_metrics"]) == 12
print("ADAPTIVE_V2_SMOKE_OK")
print("Smoke round:", smoke_results[0]["round"])
print("Smoke family:", smoke_results[0]["specification"].attack_family)
print("All-family probes:", len(smoke_results[0]["detection"]["all_family_metrics"]))

[seed=20260821] starting 1-round run with family plan: ['account_takeover']
[seed=20260821] round 1/1 starting
[seed=20260821] round 1 Agent1.research complete in 8.72s
[seed=20260821] round 1 Agent2.specify complete in 16.90s
[seed=20260821] round 1 train/unseen generation complete in 0.00s
[seed=20260821] round 1 probe 1/12 family=account_takeover rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 2/12 family=trusted_device rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 3/12 family=beneficiary_manipulation rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 4/12 family=low_and_slow rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 5/12 family=social_engineering rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 6/12 family=merchant_abuse rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 7/12 family=cross_channel_anomaly rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 8/12 family=trusted_device_normal_velocity rows=10 elapsed=0.00s
[seed=20260821] round 1 probe 9/12 fam

In [ ]:
import contextlib
import io

class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)
    def flush(self):
        for stream in self.streams:
            stream.flush()

artifacts_dir = project_dir / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)
log_path = artifacts_dir / f"adaptive_v2_run_{run_stamp}.log"

with log_path.open("w", encoding="utf-8") as log_file, contextlib.redirect_stdout(Tee(sys.stdout, log_file)), contextlib.redirect_stderr(Tee(sys.stderr, log_file)):
    print(f"LOG_SAVED {log_path}")
    print(f"RUN_CONFIGURATION seeds=1 rounds={RUN_ROUNDS} generator=conditional_ctgan detector=static family_probe_backend={FAMILY_PROBE_BACKEND} family_probe_size={FAMILY_PROBE_SIZE}")
    try:
        suite = loop.run_robustness_suite(seeds=1, rounds=RUN_ROUNDS)
    finally:
        loop.close()

print("One-seed adaptive v2 run completed")
print("Log:", log_path)

LOG_SAVED /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_run_20260825T143148Z.log
RUN_CONFIGURATION seeds=1 rounds=50 generator=conditional_ctgan detector=static
[robustness] preparing CTGAN training corpus
[robustness] fitting CTGAN: rows=880 epochs=3


/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] CTGAN ready; entering round loop
[robustness] starting suite: seeds=1, rounds=50


KeyboardInterrupt: 

In [ ]:
def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return to_jsonable(value.model_dump())
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value

artifact = {
    "run_timestamp_utc": run_stamp,
    "experiment": "adaptive_v2_qwen_ctgan_static_detector",
    "agent_backend": type(agents).__name__,
    "generator_backend": "conditional_ctgan",
    "detector_mode": "static",
    "seed_count": suite["seed_count"],
    "rounds": suite["rounds"],
    "summary": suite["summary"],
    "by_seed": suite["by_seed"],
}
result_path = artifacts_dir / f"adaptive_v2_results_{run_stamp}.json"
result_path.write_text(json.dumps(to_jsonable(artifact), indent=2), encoding="utf-8")

sys.path.insert(0, str(project_dir))
from adaptive.generate_family_charts import build_charts
build_charts(result_path, project_dir / "adaptive" / "charts")

print("RESULTS_SAVED", result_path)
print("LOG_SAVED", log_path)
print("CHARTS_SAVED", project_dir / "adaptive" / "charts")
print("ADAPTIVE_V2_KAGGLE_RUN_OK", suite["seed_count"], suite["rounds"])